## Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Load Dataset

In [2]:
df = pd.read_csv("../data/final_feature_dataset.csv")

print(df.shape)

df.head()

(3500, 16)


,ResumeID,Category,Skills,Education,Experience,Clean_Text,Text,Source,Resume_Length,Word_Count,Sentence_Count,Skill_Count,Education_Length,Experience_Length,Source_Encoded,Category_Encoded
0,REAL_0001,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...,jessica claire montgomery street san francisco...,ResumeAtlas,1495,189,1,4,3,64,0,17
1,REAL_0002,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jared arthur maica java developer 17994568777 ...,jared arthur maica java developer linkedincomi...,jared arthur maica java developer 17994568777 ...,ResumeAtlas,1686,206,1,4,3,62,0,17
2,REAL_0003,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire 9 resumesampleexamplecom montgo...,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas,5555,715,1,4,3,62,0,17
3,REAL_0004,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire 9 resumesampleexamplecom 555 43...,jessica claire 9 resumesampleexamplecom montgo...,jessica claire 9 resumesampleexamplecom 555 43...,ResumeAtlas,12834,1657,1,4,3,61,0,17
4,REAL_0005,Java Developer,"Python, SQL, Git, Linux",Computer Science degree,jessica claire 100 montgomery st 10th floor xx...,jessica claire 100 montgomery st 10th floor xx...,jessica claire 100 montgomery st 10th floor xx...,ResumeAtlas,4181,489,1,4,3,57,0,17


## Load Model

In [3]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Model Loaded Successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Loaded Successfully


## Select Resume

In [4]:
resume = df.loc[0, "Clean_Text"]

print(resume[:500])

jessica claire montgomery street san francisco ca resumesampleexamplecom professional summary highly skilled software development professional bringing 10 years software design development integration advanced knowledge java skills agile html xml jdbc tomcat work history senior java developertech lead 2014 current synnex corporation tracy ca java developer agile scrum team javascript java develop customer facing internal web applications underlying component applications wrote maintainable exten


## Job Description

In [5]:
job_description = """
Looking for a Python Developer.

Skills:

Python

Machine Learning

SQL

Git

Docker

REST API

AWS

Good communication skills.
"""

print(job_description)


Looking for a Python Developer.

Skills:

Python

Machine Learning

SQL

Git

Docker

REST API

AWS

Good communication skills.



## Semantic Score Function

In [6]:
def semantic_score(resume, jd):

    resume_embedding = model.encode(
        resume,
        convert_to_numpy=True
    )

    jd_embedding = model.encode(
        jd,
        convert_to_numpy=True
    )

    similarity = cosine_similarity(
        [resume_embedding],
        [jd_embedding]
    )

    return similarity[0][0] * 40

## Resume Quality Function

In [8]:
def resume_quality_score(resume):

    words = len(resume.split())

    if words >= 600:

        return 5

    elif words >= 400:

        return 4

    elif words >= 250:

        return 3

    elif words >= 100:

        return 2

    else:

        return 1

## Calculate Scores

In [9]:
semantic = semantic_score(
    resume,
    job_description
)

quality = resume_quality_score(
    resume
)

print("Semantic Score :", round(semantic,2))

print("Resume Quality :", quality)

Semantic Score : 11.88
Resume Quality : 2


## Temporary ATS Score

In [10]:
ats_score = semantic + quality

print("Current ATS Score :", round(ats_score,2), "/45")

Current ATS Score : 13.88 /45


## Convert to Percentage

In [11]:
ats_percentage = (ats_score / 45) * 100

print(f"ATS Percentage : {ats_percentage:.2f}%")

ATS Percentage : 30.84%


## ATS Level

In [12]:
if ats_percentage >= 85:

    print("Excellent Resume")

elif ats_percentage >= 70:

    print("Good Resume")

elif ats_percentage >= 50:

    print("Average Resume")

else:

    print("Needs Improvement")

Needs Improvement


## Summary

In [13]:
print("="*50)

print("Semantic Score :", round(semantic,2), "/40")

print("Resume Quality :", quality, "/5")

print("Current ATS :", round(ats_score,2), "/45")

print("="*50)

Semantic Score : 11.88 /40
Resume Quality : 2 /5
Current ATS : 13.88 /45


## Skills Extraction Function

In [14]:
import re

def extract_skills(text):

    skills = [
        "python","java","sql","machine learning",
        "deep learning","docker","aws","git",
        "flask","django","tensorflow","pytorch",
        "linux","kubernetes","rest api","react",
        "html","css","javascript","mongodb"
    ]

    text = text.lower()

    found_skills = []

    for skill in skills:

        if re.search(r"\b" + re.escape(skill) + r"\b", text):
            found_skills.append(skill)

    return found_skills

## Skills Match Score

In [15]:
def skill_match_score(resume, jd):

    resume_skills = set(extract_skills(resume))

    jd_skills = set(extract_skills(jd))

    matched = resume_skills.intersection(jd_skills)

    if len(jd_skills) == 0:
        return 0, matched

    score = (len(matched) / len(jd_skills)) * 30

    return score, matched

## Calculate Skills Score

In [16]:
skill_score, matched_skills = skill_match_score(
    resume,
    job_description
)

print("Matched Skills :", matched_skills)

print("Skill Score :", round(skill_score,2), "/30")

Matched Skills : set()
Skill Score : 0.0 /30


## Experience Score

In [18]:
def experience_score(resume):

    resume = resume.lower()

    experience_keywords = [
        "experience",
        "worked",
        "developer",
        "engineer",
        "intern",
        "project"
    ]

    count = 0

    for word in experience_keywords:

        if word in resume:
            count += 1

    score = min(count * 2.5, 15)

    return score

## Education Score

In [20]:
def education_score(resume):

    resume = resume.lower()

    education_keywords = [
        "b.tech",
        "bachelor",
        "master",
        "computer science",
        "engineering",
        "university"
    ]

    count = 0

    for word in education_keywords:

        if word in resume:
            count += 1

    score = min(count * 2,10)

    return score

## Calculate All Scores

In [21]:
experience = experience_score(resume)

education = education_score(resume)

print("Experience :", experience,"/15")

print("Education :", education,"/10")

Experience : 7.5 /15
Education : 4 /10


## Final ATS Score

In [23]:
final_ats = (
    semantic +
    quality +
    skill_score +
    experience +
    education
)

print(f"Final ATS Score : {final_ats:.2f}/100")

Final ATS Score : 25.38/100


## ATS Rating

In [25]:
if final_ats >= 90:

    level = "Excellent"

elif final_ats >= 75:

    level = "Good"

elif final_ats >= 60:

    level = "Average"

else:

    level = "Needs Improvement"

print("Resume Rating :", level)

Resume Rating : Needs Improvement


## Score Card

In [27]:
score_card = {
    "Semantic Match": round(semantic,2),
    "Skills Match": round(skill_score,2),
    "Experience": experience,
    "Education": education,
    "Resume Quality": quality,
    "Final ATS": round(final_ats,2)
}

score_card

{'Semantic Match': np.float32(11.88),
 'Skills Match': 0.0,
 'Experience': 7.5,
 'Education': 4,
 'Resume Quality': 2,
 'Final ATS': np.float32(25.38)}

## Save ATS Report

In [28]:
import os

os.makedirs("../outputs", exist_ok=True)

report = pd.DataFrame([score_card])

report.to_csv(
    "../outputs/ats_report.csv",
    index=False
)

print("ATS Report Saved Successfully")

ATS Report Saved Successfully
